In [1]:
import glob
import os
import pandas as pd
from rapidfuzz import process, fuzz

In [2]:
COLS = {
    'ieeexplore': ('Document Title', 'DOI'),
    'sciencedirect': ('title', 'doi'),
    'springer': ('Item Title', 'Item DOI'),
    'wiley': ('title', 'doi'),
    'acm':  ('title', 'doi'),
}

def norm_doi(s: str):
    s = s.str.strip().str.lower()
    s = s.str.replace(r'^(https?://(dx\.)?doi\.org/|doi:)', '', regex=True)
    return s.replace('', pd.NA)

def join_uniq(s):
    return ";".join(sorted(set(s.dropna())))

In [3]:
# read and merge file by DOI
frames = []

for f in sorted(glob.glob('data/ss*.csv')):
    name = os.path.basename(f)
    search_string = name.split('-')[0]
    repo = name.split('-')[1]
    tcol, dcol = COLS[repo]
    df = pd.read_csv(f, dtype=str, engine='python', on_bad_lines='warn')
    out = pd.DataFrame({
        'title': df[tcol],
        'doi': norm_doi(df[dcol]),
        'search_string': search_string,
        'repository': repo,
    })
    frames.append(out)

C:\Users\tungo\AppData\Local\Temp\ipykernel_42740\2685799375.py:9: ParserWarning: Skipping line 850: Expected 10 fields in line 850, saw 19

  df = pd.read_csv(f, dtype=str, engine='python', on_bad_lines='warn')
C:\Users\tungo\AppData\Local\Temp\ipykernel_42740\2685799375.py:9: ParserWarning: Skipping line 1810: Expected 10 fields in line 1810, saw 19

  df = pd.read_csv(f, dtype=str, engine='python', on_bad_lines='warn')
C:\Users\tungo\AppData\Local\Temp\ipykernel_42740\2685799375.py:9: ParserWarning: Skipping line 2556: Expected 10 fields in line 2556, saw 19

  df = pd.read_csv(f, dtype=str, engine='python', on_bad_lines='warn')
C:\Users\tungo\AppData\Local\Temp\ipykernel_42740\2685799375.py:9: ParserWarning: Skipping line 1135: field larger than field limit (131072)

  df = pd.read_csv(f, dtype=str, engine='python', on_bad_lines='warn')


In [4]:
allrows = pd.concat(frames, ignore_index=True)
allrows['doi_key'] = allrows['doi']
filtered = allrows[allrows['doi_key'].notna()]
nodoi_rows = allrows[allrows['doi_key'].isna()].copy()

In [5]:
grouped = filtered.groupby('doi_key', as_index=False).agg(
    title=('title', 'first'),
    doi=('doi', 'first'),
    search_strings=('search_string', join_uniq),
    repositories=('repository', join_uniq),
).drop(columns='doi_key')

nodoi_rows = nodoi_rows.rename(columns={'search_string': 'search_strings', 'repository': 'repositories'})
nodoi_rows = nodoi_rows[['title', 'doi', 'search_strings', 'repositories']]

merged = pd.concat([grouped, nodoi_rows], ignore_index=True)

print(f'input rows : {len(allrows)}')
print(f'unique DOIs: {len(grouped)}')
print(f'no-DOI kept: {len(nodoi_rows)}')
print(f'output rows: {len(merged)}')

merged.to_csv('out/merged_search_results.csv', index=False)

input rows : 17749
unique DOIs: 15589
no-DOI kept: 325
output rows: 15914


In [6]:
final = pd.read_csv(
    'data/Final Set- WithQAScores- WithoutStudyID(Remove Duplicated).csv',
    dtype=str, engine='python', on_bad_lines='warn'
)

# match on doi
final['doi_key'] = norm_doi(final['DOI'])

matched_doi = final.merge(
    merged.assign(doi_key=merged['doi']),
    on='doi_key', how='inner', suffixes=('_final', '_merged'),
)

# match on title (fuzz)
choices = merged['title'].dropna()
choice_list = choices.tolist()
choice_idx = choices.index.tolist()

rows = []
for fi, ftitle in final['Title'].dropna().items():
    hit = process.extractOne(ftitle, choice_list, scorer=fuzz.token_sort_ratio)
    if hit and hit[1] > 85:
        _, score, pos = hit
        m = merged.loc[choice_idx[pos]]
        rows.append({
            'final_idx': fi,
            'final_title': ftitle,
            'merged_title': m['title'],
            'score': score,
            'doi': m['doi'],
            'search_strings': m['search_strings'],
            'repositories': m['repositories'],
        })

matched_title = pd.DataFrame(rows)
print(f'input final: {len(final)}')
print(f'matched on DOI  : {len(matched_doi)}')
print(f'matched on title: {len(matched_title)}')

matched_doi.to_csv('out/matched_doi_results.csv', index=False)
matched_title.to_csv('out/matched_title_results.csv', index=False)


input final: 95
matched on DOI  : 59
matched on title: 51


In [7]:
# explore matched result: unify both methods on StudyID
doi_part = matched_doi[['StudyID', 'title', 'search_strings', 'repositories']].copy()
doi_part['method'] = 'doi'

title_part = matched_title.copy()
title_part['StudyID'] = title_part['final_idx'].map(final['StudyID'])
title_part = title_part.rename(columns={'merged_title': 'title'})
title_part = title_part[['StudyID', 'title', 'search_strings', 'repositories']].copy()
title_part['method'] = 'title'

long = pd.concat([doi_part, title_part], ignore_index=True)
long = long[long['StudyID'].notna()]
print(f'doi rows  : {len(doi_part)}')
print(f'title rows: {len(title_part)}')
print(f'long rows : {len(long)}')

doi rows  : 59
title rows: 51
long rows : 110


In [8]:
# combined deduped matched set (one row per StudyID)
def union_split(s):
    vals = set()
    for x in s.dropna():
        vals.update(p for p in x.split(';') if p)
    return ";".join(sorted(vals))

combined = long.groupby('StudyID', as_index=False).agg(
    title=('title', 'first'),
    search_strings=('search_strings', union_split),
    repositories=('repositories', union_split),
    methods=('method', join_uniq),
)

print(f'combined matched papers: {len(combined)}')
print(combined['methods'].value_counts())

combined.to_csv('out/matched_combined.csv', index=False)

combined matched papers: 62
methods
doi;title    48
doi          11
title         3
Name: count, dtype: int64


In [9]:
# distribution of combined matched papers across repository and search string
dist = combined.assign(
    repository=combined['repositories'].str.split(';'),
    search_string=combined['search_strings'].str.split(';'),
).explode('repository').explode('search_string')

print('papers per repository')
print(dist.drop_duplicates(['StudyID', 'repository'])['repository'].value_counts(), '\n')

print('papers per search string')
print(dist.drop_duplicates(['StudyID', 'search_string'])['search_string'].value_counts(), '\n')

print('repository x search string')

papers per repository
repository
acm              28
springer         17
ieeexplore       13
sciencedirect     7
wiley             2
Name: count, dtype: int64 

papers per search string
search_string
ss6    57
ss4    26
ss2    23
Name: count, dtype: int64 

repository x search string


In [10]:
# overlap articles: matched by BOTH doi and title methods
overlap = combined[combined['methods'] == 'doi;title'].copy()
overlap = overlap[['StudyID', 'title', 'search_strings', 'repositories']]

print(f'overlap articles (both methods): {len(overlap)}')
overlap.to_csv('out/overlap_articles.csv', index=False)
overlap

overlap articles (both methods): 48


,StudyID,title,search_strings,repositories
0,11,A Simulation Framework for Workload Management...,ss6,acm;ieeexplore
1,12,A Software Platform to Support Disaggregated Q...,ss6,acm;ieeexplore
3,16,Choreography and Profiling of Quantum-Classica...,ss2;ss6,ieeexplore
4,17,CUDA Quantum: The Platform for Integrated Quan...,ss4;ss6,acm;ieeexplore
5,18,𝐶𝑙𝑎𝑠𝑠𝑖|𝑄⟩: Towards a Translation Framework to ...,ss6,acm
6,19,Developing hybrid quantum-classical software: ...,ss6,acm
7,2,: A Cross-Platform Programming Framework for Q...,ss2;ss4;ss6,springer
9,22,Extending C++ for Heterogeneous Quantum-Classi...,ss4;ss6,acm
10,23,GeQuPI: Quantum Program Improvement with Multi...,ss6,sciencedirect
11,24,HetArch: Heterogeneous Microarchitectures for ...,ss6,acm


In [11]:
list_matched_studyid = combined['StudyID'].to_list()
un_matched = final[~final['StudyID'].isin(list_matched_studyid)].copy()
un_matched.to_csv('out/un_matched_articles.csv',index=False)
